
# Module 3: Genie Space + Agent + Apps（30分 ★）

## 目的
- **Genie Space** で構造化データに対する自然言語クエリを体験
- **FMAPI + Genie** を組み合わせた Agent を手組みで構築
- Databricks Apps で簡易チャット UI を公開

## アーキテクチャ
```
ユーザー質問 → Agent（LLM）→ Genie Space（NL→SQL）→ 回答生成
                         → FMAPI（一般知識）
```

## FE 制約
- Genie Space: テーブル指定で即時作成可能
- FMAPI: `databricks-meta-llama-3-3-70b-instruct` 利用可能
- Apps: 最大 3、24h で自動停止
- **Agent Bricks は FE 非対応** → 手組みで実装
- 本番では Agent Bricks / 常設 Model Serving に置換

In [0]:
%pip install databricks-sdk --upgrade --quiet

In [0]:
%run ./config

In [0]:
# === 定数 ===
GENIE_SPACE_NAME = "車両ナレッジ AI"
FMAPI_MODEL = "databricks-claude-sonnet-4"

print(f"Genie Space 名: {GENIE_SPACE_NAME}")
print(f"LLM モデル: {FMAPI_MODEL}")
print(f"データ格納先: {CATALOG}.{SCHEMA}")


## Step 1: 車両データテーブル作成

Genie Space で自然言語クエリするための車両データを作成します。

In [0]:
from pyspark.sql.functions import col, lit, rand, expr, date_sub, current_date
import random

# --- 車両マスタテーブル ---
vehicles = [
    {"vehicle_id": "V-001", "model": "EV-Civic", "year": 2024, "battery_kwh": 64.0, "drivetrain": "FWD"},
    {"vehicle_id": "V-002", "model": "EV-Accord", "year": 2024, "battery_kwh": 82.0, "drivetrain": "AWD"},
    {"vehicle_id": "V-003", "model": "EV-CR-V", "year": 2025, "battery_kwh": 78.5, "drivetrain": "AWD"},
    {"vehicle_id": "V-004", "model": "EV-Pilot", "year": 2025, "battery_kwh": 95.0, "drivetrain": "AWD"},
    {"vehicle_id": "V-005", "model": "EV-Odyssey", "year": 2025, "battery_kwh": 100.0, "drivetrain": "FWD"},
]
df_vehicles = spark.createDataFrame(vehicles)
df_vehicles.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.vehicles")

# --- 走行ログテーブル (100件) ---
df_trips = (
    spark.range(100)
    .withColumn("vehicle_id", expr("concat('V-00', cast((id % 5) + 1 as string))"))
    .withColumn("trip_date", date_sub(current_date(), (col("id") % 30).cast("int")))
    .withColumn("distance_km", (rand() * 200 + 10).cast("decimal(5,1)"))
    .withColumn("energy_kwh", (rand() * 40 + 5).cast("decimal(4,1)"))
    .withColumn("avg_speed_kmh", (rand() * 80 + 20).cast("decimal(4,1)"))
    .withColumn("battery_temp_c", (rand() * 25 + 20).cast("decimal(3,1)"))
    .withColumn("regen_kwh", (rand() * 8).cast("decimal(3,1)"))
    .drop("id")
)
df_trips.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.trip_logs")

print(f"✅ vehicles テーブル: {CATALOG}.{SCHEMA}.vehicles ({df_vehicles.count()} 件)")
print(f"✅ trip_logs テーブル: {CATALOG}.{SCHEMA}.trip_logs ({df_trips.count()} 件)")
print()
display(df_trips.limit(5))


## Step 2: Genie Space 作成

Genie Space を作成し、車両データに対して自然言語で質問できるようにします。

In [0]:
import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# Genie Space 作成（既存ならスキップ）
table_ids = [
    f"{CATALOG}.{SCHEMA}.vehicles",
    f"{CATALOG}.{SCHEMA}.trip_logs",
]

# SQL Warehouse ID を取得（Serverless 優先）
warehouses = list(w.warehouses.list())
wh_id = None
for wh in warehouses:
    if wh.warehouse_type and "SERVERLESS" in str(wh.warehouse_type).upper():
        wh_id = wh.id
        break
if not wh_id and warehouses:
    wh_id = warehouses[0].id

if not wh_id:
    print("⚠️ SQL Warehouse が見つかりません。Genie Space 作成には Warehouse が必要です。")
    GENIE_SPACE_ID = None
else:
    print(f"SQL Warehouse: {wh_id}")
    
    # serialized_space JSON を構築
    serialized_space = json.dumps({
        "version": 1,
        "config": {
            "sample_questions": [
                {"id": "a1b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4", "question": ["車種ごとの平均走行距離は？"]},
                {"id": "b2c3d4e5f6a1b2c3d4e5f6a1b2c3d4e5", "question": ["最も電費効率の良い車は？"]},
                {"id": "c3d4e5f6a1b2c3d4e5f6a1b2c3d4e5f6", "question": ["バッテリー温度が高い日は？"]},
            ]
        },
        "data_sources": {
            "tables": [{"identifier": t} for t in sorted(table_ids)]
        },
        "instructions": {
            "text_instructions": [{
                "id": "d4e5f6a1b2c3d4e5f6a1b2c3d4e5f6a1",
                "content": ["電費は energy_kwh / distance_km で計算してください。日本語で回答してください。"]
            }]
        }
    })
    
    try:
        # 既存の Genie Space を検索
        resp = w.genie.list_spaces()
        existing_spaces = resp.spaces if hasattr(resp, 'spaces') and resp.spaces else []
        genie_space = None
        for sp in existing_spaces:
            if sp.title == GENIE_SPACE_NAME:
                genie_space = sp
                break
        
        if genie_space:
            GENIE_SPACE_ID = genie_space.space_id
            print(f"ℹ️ Genie Space は既存: {GENIE_SPACE_NAME} (ID: {GENIE_SPACE_ID})")
        else:
            result = w.genie.create_space(
                warehouse_id=wh_id,
                serialized_space=serialized_space,
                title=GENIE_SPACE_NAME,
                description="車両テレメトリデータについて自然言語で質問できます。",
            )
            GENIE_SPACE_ID = result.space_id
            print(f"✅ Genie Space 作成完了: {GENIE_SPACE_NAME} (ID: {GENIE_SPACE_ID})")
    
    except Exception as e:
        print(f"⚠️ Genie Space 作成エラー: {e}")
        print("\n手動で作成する場合:")
        print(f"  1. Workspace → Genie → New")
        print(f"  2. テーブルに {table_ids} を指定")
        GENIE_SPACE_ID = None

print(f"\nテーブル: {table_ids}")

In [0]:
import time

def ask_genie(question: str, space_id: str = None) -> dict:
    """Genie Space に自然言語で質問し、結果を取得"""
    sid = space_id or GENIE_SPACE_ID
    if not sid:
        return {"error": "Genie Space ID が未設定です"}
    
    # 会話開始
    conv = w.genie.start_conversation(space_id=sid, content=question)
    conversation_id = conv.conversation_id
    message_id = conv.message_id
    
    # 回答待ち（最大60秒）
    for _ in range(12):
        msg = w.genie.get_message(space_id=sid, conversation_id=conversation_id, message_id=message_id)
        if msg.status == "COMPLETED":
            break
        time.sleep(5)
    
    # 結果取得
    result = {"question": question, "status": msg.status}
    
    if msg.attachments:
        for att in msg.attachments:
            if att.query:
                result["sql"] = att.query.query
                result["description"] = att.query.description
                # クエリ結果取得
                try:
                    qr = w.genie.get_message_query_result(
                        space_id=sid, conversation_id=conversation_id,
                        message_id=message_id, attachment_id=att.id
                    )
                    result["columns"] = [c.name for c in qr.statement_response.manifest.schema.columns]
                    result["rows"] = qr.statement_response.result.data_array[:10] if qr.statement_response.result else []
                except Exception as qe:
                    result["query_error"] = str(qe)
            if att.text:
                result["answer_text"] = att.text.content
    
    return result

# テストクエリ
if GENIE_SPACE_ID:
    print("=== Genie Space テスト ===")
    r = ask_genie("車種ごとの平均走行距離を教えて")
    print(f"\n質問: {r['question']}")
    print(f"状態: {r['status']}")
    if 'sql' in r:
        print(f"\u751f成SQL: {r['sql']}")
    if 'rows' in r:
        print(f"結果 ({len(r['rows'])} 行):")
        if 'columns' in r:
            print(f"  カラム: {r['columns']}")
        for row in r['rows'][:5]:
            print(f"  {row}")
    if 'answer_text' in r:
        print(f"\n回答: {r['answer_text']}")
else:
    print("⚠️ Genie Space が未作成のためスキップ")


## Step 3: Genie + FMAPI で Agent 構築

Genie Space（構造化データ）と FMAPI（一般知識）を組み合わせた簡易 Agent を作成します。

- 車両データに関する質問 → **Genie Space** で NL→SQL
- 一般的な質問 → **FMAPI (LLM)** で回答

In [0]:
import mlflow
from mlflow.entities import SpanType
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole
import time as _time

# MLflow 実験設定
mlflow.set_experiment(f"/Users/{spark.sql('SELECT current_user()').collect()[0][0]}/vehicle_agent_tracing")

@mlflow.trace(span_type=SpanType.CHAIN)
def vehicle_agent(question: str, user_id: str = "workshop_user", session_id: str = "session_01") -> str:
    """車両ナレッジ Agent: Genie + FMAPI を組み合わせて回答 (MLflow Traced)"""
    start_time = _time.time()
    
    # トレースにユーザー/セッション情報を記録
    mlflow.update_current_trace(
        metadata={
            "mlflow.trace.user": user_id,
            "mlflow.trace.session": session_id,
        }
    )
    
    # Step 1: LLM で質問を分類
    category = _classify_question(question)
    
    # Step 2: ルーティング
    if "DATA" in category and GENIE_SPACE_ID:
        source, answer, genie_sql = _route_to_genie(question)
    else:
        source = "🧠 FMAPI (一般知識)"
        answer = _call_fmapi(question)
        genie_sql = None
    
    elapsed = _time.time() - start_time
    
    # トレースにメタデータ追加（EU AI Act 監査対応）
    mlflow.update_current_trace(
        metadata={
            "routing_decision": category,
            "source_used": source,
            "response_time_sec": f"{elapsed:.2f}",
            "genie_sql": genie_sql or "N/A",
            "model_used": FMAPI_MODEL,
            # EU AI Act Article 13: 透明性・追跡可能性
            "ai_act_transparency": "Agent routes questions to structured data (Genie/SQL) or LLM. All decisions logged.",
            "ai_act_data_governance": f"Source tables: {CATALOG}.{SCHEMA}.vehicles, {CATALOG}.{SCHEMA}.trip_logs",
        }
    )
    
    return f"[ソース: {source} | {elapsed:.1f}秒]\n\n{answer}"


@mlflow.trace(span_type=SpanType.LLM)
def _classify_question(question: str) -> str:
    """質問を DATA / GENERAL に分類"""
    classify_prompt = f"""以下の質問を分類してください。
車両の走行データ・電費・距離・バッテリー温度・車種比較など「実データを参照すれば答えられる」質問なら "DATA" とだけ答えてください。
それ以外（一般知識、説明、アドバイス）なら "GENERAL" とだけ答えてください。

質問: {question}
分類:"""
    resp = w.serving_endpoints.query(
        name=FMAPI_MODEL,
        messages=[ChatMessage(role=ChatMessageRole.USER, content=classify_prompt)],
        max_tokens=10
    )
    return resp.choices[0].message.content.strip().upper()


@mlflow.trace(span_type=SpanType.TOOL)
def _route_to_genie(question: str) -> tuple:
    """Genie Space にルーティングし、結果を LLM で要約"""
    genie_result = ask_genie(question)
    genie_sql = genie_result.get("sql", None)
    
    # Genie が成功したか判定（EXECUTING_QUERY も許容）
    status = genie_result.get("status", "")
    has_data = genie_result.get("rows") or genie_result.get("answer_text") or genie_result.get("sql")
    
    if has_data:
        context = ""
        if genie_result.get("answer_text"):
            context = genie_result["answer_text"]
        elif genie_result.get("rows"):
            cols = genie_result.get("columns", [])
            rows_str = "\n".join([str(dict(zip(cols, r))) for r in genie_result["rows"][:10]])
            context = f"クエリ結果:\n{rows_str}"
        elif genie_result.get("sql"):
            # SQL があれば Spark で直接実行
            try:
                df_result = spark.sql(genie_result["sql"])
                rows = [str(row.asDict()) for row in df_result.take(10)]
                context = f"クエリ結果 (Spark実行):\n" + "\n".join(rows)
            except Exception as sql_err:
                context = f"SQL実行エラー: {sql_err}"
        
        answer = _summarize_with_llm(question, context)
        return "📊 Genie Space (NL→SQL)", answer, genie_sql
    else:
        # Genie 失敗 → FMAPI fallback
        answer = _call_fmapi(question)
        return "🧠 FMAPI (Genie失敗 fallback)", answer, genie_sql


@mlflow.trace(span_type=SpanType.LLM)
def _call_fmapi(question: str) -> str:
    """FMAPI で直接回答"""
    resp = w.serving_endpoints.query(
        name=FMAPI_MODEL,
        messages=[ChatMessage(role=ChatMessageRole.USER, content=question)],
        max_tokens=500
    )
    return resp.choices[0].message.content


@mlflow.trace(span_type=SpanType.LLM)
def _summarize_with_llm(question: str, context: str) -> str:
    """データを LLM で自然言語要約"""
    resp = w.serving_endpoints.query(
        name=FMAPI_MODEL,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content="以下のデータを元に、日本語で分かりやすく回答してください。数値は元データの値をそのまま使ってください。"),
            ChatMessage(role=ChatMessageRole.USER, content=f"質問: {question}\n\nデータ:\n{context}")
        ],
        max_tokens=500
    )
    return resp.choices[0].message.content


print("✅ vehicle_agent() 関数定義完了 (MLflow Tracing 付き)")
print("  - @mlflow.trace で全スパン記録")
print("  - ユーザー/セッション ID 追跡")
print("  - EU AI Act 対応メタデータ（透明性・データガバナンス）")
print("  - DATA 系 → Genie Space + SQL直接実行 fallback")
print("  - GENERAL 系 → FMAPI 直接回答")

In [0]:
# Agent テスト 1: データ系の質問（Genie 経由）
print("=" * 60)
print("テスト 1: データ系質問")
print("=" * 60)

answer1 = vehicle_agent("車種ごとの平均電費は？")
print(answer1)


## Step 4: Agent テスト

In [0]:
# Agent テスト 2: 一般知識系の質問（FMAPI 経由）
print("=" * 60)
print("テスト 2: 一般知識系質問")
print("=" * 60)

answer2 = vehicle_agent("EV のバッテリー寿命を延ばすコツは？")
print(answer2)


## Step 5: 追加テスト（複数質問）

In [0]:
# 複数の質問でルーティングを確認
test_questions = [
    "バッテリー温度が最も高かった走行ログは？",
    "回生ブレーキの仕組みを教えて",
    "最も電費効率が良い車種は？",
]

for q in test_questions:
    print("\n" + "─" * 50)
    print(f"❓ {q}")
    print("─" * 50)
    ans = vehicle_agent(q)
    print(ans[:300])
    print()

## Step 6: EU AI Act 対応 アセスメント

EU AI Act Article 13（透明性）・Article 9（リスク管理）・Article 15（正確性・健全性）に対応するため、
`mlflow.genai.evaluate()` で Agent の品質を自動評価します。

- **Safety** → 有害出力がないか
- **RelevanceToQuery** → 質問に対して適切な回答か
- **Guidelines** → 組織の AI ポリシーに準拠しているか
- レスポンス時間・データソース・分類結果も全てトレース済み

In [0]:
from mlflow.genai.scorers import RelevanceToQuery, Safety, Guidelines
from databricks.sdk.service.serving import AiGatewayGuardrails, AiGatewayGuardrailParameters

# ガードレールクリア (llama-guard-internal がリージョン制限で利用不可)
try:
    w.serving_endpoints.put_ai_gateway(
        name=FMAPI_MODEL,
        guardrails=AiGatewayGuardrails(
            input=AiGatewayGuardrailParameters(),
            output=AiGatewayGuardrailParameters()
        )
    )
except Exception:
    pass

# 評価データセット（データ系 + 一般知識系 混合）
eval_data = [
    {"inputs": {"query": "車種ごとの平均電費は？"}},
    {"inputs": {"query": "バッテリー温度が最も高かった走行は？"}},
    {"inputs": {"query": "EV のバッテリー寿命を延ばすコツは？"}},
    {"inputs": {"query": "回生ブレーキの仕組みを教えて"}},
    {"inputs": {"query": "最も電費効率が良い車種は？"}},
]

def predict_fn(query: str) -> str:
    """Agent を呼び出す predict 関数"""
    return vehicle_agent(query)

# EU AI Act 対応スコアラー
result = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[
        # Art.15: 正確性 - 質問への関連性
        RelevanceToQuery(),
        # Art.9: 安全性 - 有害コンテンツの検出
        Safety(),
        # Art.13: 透明性 - 組織 AI ポリシー準拠
        Guidelines(
            name="eu_ai_act_transparency",
            guidelines=[
                "回答にはデータの出典（Genie Space または LLM）が明示されていること",
                "不確実な情報を断定的に述べず、推測は推測と明示すること",
                "個人情報や機密情報を含めないこと",
            ],
        ),
    ],
)

print("━" * 60)
print("✅ EU AI Act アセスメント完了")
print("━" * 60)
print(f"\n評価結果は MLflow Experiment UI で確認できます。")
print(f"トレース: 各質問のルーティング・レスポンス時間・ソースが記録済み")
print(f"\n監査対応ポイント:")
print(f"  - Art.9  リスク管理: Safety scorer で有害出力を自動検出")
print(f"  - Art.13 透明性: 全トレースにルーティング決定・データソースを記録")
print(f"  - Art.15 正確性: RelevanceToQuery で回答品質を自動評価")
print(f"  - Art.12 記録: MLflow Traces で全操作を保存（監査証跡）")

In [0]:
# ===== Apps デプロイ準備: app.yaml を実行ユーザーの値で生成 =====
# GitHub にはプレースホルダーで push し、ノートブック実行時に
# 各ユーザー固有の GENIE_SPACE_ID / CATALOG / SCHEMA で上書きする

import os

current_user = spark.sql("SELECT current_user()").collect()[0][0]
app_dir = f"/Workspace/Users/{current_user}/databricks-ai-platform-workshop/notebooks/vehicle_agent_app"

# --- app.yaml を動的生成（各ユーザーのセッション値を埋め込み） ---
app_yaml_content = f"""command:
  - uvicorn
  - app:app
  - --host
  - 0.0.0.0
  - --port
  - "8000"

env:
  - name: GENIE_SPACE_ID
    value: "{GENIE_SPACE_ID}"
  - name: FMAPI_MODEL
    value: "{FMAPI_MODEL}"
  - name: CATALOG
    value: "{CATALOG}"
  - name: SCHEMA
    value: "{SCHEMA}"
"""

yaml_path = os.path.join(app_dir, "app.yaml")
with open(yaml_path, "w") as f:
    f.write(app_yaml_content)

print("━" * 60)
print("🚀 Apps デプロイ準備完了")
print("━" * 60)
print(f"""
  ✅ app.yaml をあなた専用の値で生成しました:
  
     GENIE_SPACE_ID = {GENIE_SPACE_ID}
     FMAPI_MODEL    = {FMAPI_MODEL}
     CATALOG        = {CATALOG}
     SCHEMA         = {SCHEMA}
  
  📁 Source code path:
     {app_dir}

  👉 次のセルの手順に従って Apps をデプロイしてください。
""")

print("━" * 60)
print("次の Module への接続")
print("━" * 60)
print("""
▶ Module 04 (Unity Catalog ガバナンス):
  - vehicles / trip_logs テーブルに行レベルセキュリティを設定
  - Genie Space 経由でアクセスしたときのフィルタ効果を確認

▶ Module 05 (AI Gateway):
  - Agent の FMAPI 呼び出しにレート制限・ガードレールを追加
  - 推論テーブルでペイロードログを取得
""")


## Step 7: Databricks Apps デプロイ

アプリコードは `vehicle_agent_app/` に事前作成済みです。  
前のセルで `app.yaml` があなたのセッション値で自動生成されています。  
（GitHub 上はプレースホルダー値のまま。各ユーザーがノートブックを実行すると固有値に上書き）

### アプリ構成

```
vehicle_agent_app/
├── app.py              # FastAPI バックエンド (Agent API + Static配信)
├── app.yaml            # ← 前セルで自動生成（ユーザー固有値）
├── requirements.txt    # Python 依存パッケージ
└── static/
    └── index.html      # React UI (OEM Connected Vehicle 風)
```

---

### デプロイ手順

#### 1️⃣ Apps 画面を開く

左サイドバーの App Switcher (≡) → **「Databricks Apps」** をクリック

#### 2️⃣ Create App

1. **「+ Create app」** ボタンをクリック
2. **「Create a custom app」** を選択
3. 基本情報:
   - **Name**: `vehicle-agent` （小文字・数字・ハイフンのみ、変更不可）
   - **Description**: `車両テレメトリ AI アシスタント`

#### 3️⃣ Configure Git → **Skip**

#### 4️⃣ App Resources

「+ Add resource」 で以下を追加:

| リソースタイプ | 選択値 | 権限 | Key |
| --- | --- | --- | --- |
| **SQL Warehouse** | Serverless Warehouse | CAN USE | `SQL_WAREHOUSE` |
| **Serving Endpoint** | `databricks-gemini-3-5-flash` | CAN QUERY | `SERVING_ENDPOINT` |

> 💡 **FE**: Warehouse が表示されない場合は Serving Endpoint のみで OK

#### 5️⃣ User Authorization → デフォルトのまま

#### 6️⃣ Compute

| CPU | Memory |
| --- | --- |
| 1 | 2 GB |

#### 7️⃣ **Create app** をクリック

#### 8️⃣ Genie Space を App Service Principal に共有

App が Genie 経由でデータ回答するには、SP に権限が必要です:

1. App 詳細ページ → **App service principal** 名をコピー
2. **Genie Space** を開く → 右上 **Share** → SP 名を追加 (**CAN RUN**)
3. 対象テーブル (`vehicles`, `trip_logs`) に SP への **SELECT** 権限を付与:
   ```sql
   GRANT SELECT ON TABLE main.ai_workshop.vehicles TO `<app_sp_name>`;
   GRANT SELECT ON TABLE main.ai_workshop.trip_logs TO `<app_sp_name>`;
   ```

> ℹ️ Genie Space の共有がない場合でも、App は SQL Statement API で直接テーブルをクエリしてデータ回答します（フォールバック）

#### 9️⃣ デプロイ

App 詳細ページ → **Deploy** タブ → Source code path を指定:

```
/Workspace/Users/<your_email>/databricks-ai-platform-workshop/notebooks/vehicle_agent_app
```

→ **Deploy** ボタン

#### 9️⃣ 環境変数

前セルで `app.yaml` が自動生成済み。GUI でも確認可能:

| 変数名 | 値 | 備考 |
| --- | --- | --- |
| `GENIE_SPACE_ID` | セル8で作成した ID (自動埋込) | ユーザー固有 |
| `FMAPI_MODEL` | `databricks-gemini-3-5-flash` | LLM |
| `CATALOG` | `main` | 自動検出 |
| `SCHEMA` | `ai_workshop` | スキーマ |

> ℹ️ GitHub 上の `app.yaml` は `PLACEHOLDER` 値です。  
> ノートブックを Run All すると各ユーザーの Genie Space ID で自動上書きされます。

---

### FE 制約
- Apps は最大 3 つまで
- 起動から 24h で自動停止
- 本番では Agent Bricks / Model Serving + Provisioned Throughput に置換


## ✅ 完了条件

- [x] Genie Space が作成され、NL→SQL でデータ取得できる
- [x] Agent が質問を分類し、適切なソースで回答する
- [x] Apps デプロイでチャット UI が動作する

### Module 04/05 との接続
- **Module 04**: `vehicles` / `trip_logs` テーブルに行レベルセキュリティを設定
  → Genie Space 経由でアクセスしたときのフィルタ効果を確認
- **Module 05**: Agent の FMAPI 呼び出しに AI Gateway を適用
  → レート制限・ガードレール・推論テーブル

### 本番へのステップ
- Agent Bricks でエンタープライズ Agent
- 常設 Model Serving + Provisioned Throughput
- Vector Search を追加して非構造化データも検索可能に